# Fase 3 — Cleaning + Deduplikasi

**Input :** `data/interim/news_clean_raw.parquet` (artikel kandidat hasil crawl)
**Output:** `data/interim/news_clean.parquet` (teks bersih, bebas duplikat)

Langkah: normalisasi → buang boilerplate → filter (foto/video, advertorial, <40 kata)
→ dedup eksak → dedup near-duplicate (MinHash LSH).

> Butuh `datasketch` (untuk near-dup). Install bila belum:
> `pip install datasketch` — sudah ada di requirements-crawl.txt versi terbaru.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import yaml

if Path('src').exists():
    ROOT = Path('.').resolve()
elif Path('../src').exists():
    os.chdir('..'); ROOT = Path('.').resolve()
import sys; sys.path.insert(0, str(ROOT))
from src.clean import clean_news, CleanConfig

cfg = yaml.safe_load(open(ROOT / 'config/config.yaml'))
INTERIM = ROOT / 'data/interim'
raw = pd.read_parquet(INTERIM / 'news_clean_raw.parquet')
print('artikel masuk:', len(raw))

In [ ]:
cc = CleanConfig(
    min_words=cfg['filters']['min_words'],
    exclude_subchannels=tuple(cfg['filters']['exclude_subchannels']),
    boilerplate_path=ROOT / 'config/boilerplate_patterns.txt',
    near_dup_threshold=0.8,
    near_dup_window_days=2,
)

clean, stats = clean_news(raw, cc)

print('===== STATISTIK CLEANING =====')
for k, v in stats.items():
    print(f'  {k:20s}: {v}')

In [ ]:
# Simpan output
out_path = INTERIM / 'news_clean.parquet'
clean.to_parquet(out_path, index=False)
print('tersimpan:', out_path.resolve())
print('kolom    :', list(clean.columns))
clean[['article_id', 'published_at', 'subchannel', 'n_words', 'title']].head()

In [ ]:
# QC cepat: pastikan boilerplate hilang + lihat distribusi panjang
for pat in ['Gambas', 'Baca juga', 'ADVERTISEMENT']:
    n = clean.body_text.str.contains(pat, case=False, regex=False).sum()
    print(f'sisa "{pat}": {n}  (harus 0)')
print('\nn_words: min', clean.n_words.min(), '| median', int(clean.n_words.median()), '| max', clean.n_words.max())
print('artikel per tahun:')
print(clean.published_at.str[:4].value_counts().sort_index().to_string())

## Setelah selesai
Copy-paste output **STATISTIK CLEANING** dan **QC** ke chat untuk verifikasi,
lalu lanjut ke Fase 4 (Tiering relevansi per bank).